In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Cytolysis dataset integration and label-consistency analysis

- This notebook performs sequence-level integration and curation of cytolysis annotations using AMPDB v1 as input. The primary input consists of a processed cytolysis dataset (processed_cytolysis_dataset.csv) containing peptide sequences and their associated labels from a single source.

- The workflow begins by extracting all unique peptide sequences and constructing a pivot table where each row represents a unique sequence and each column corresponds to a data source. The pivot structure enables systematic comparison and aggregation of annotations at the sequence level.

- Quality control is applied by removing sequences containing non-canonical amino acids and by enforcing minimum and maximum length constraints defined in the global configuration. These filters ensure that the resulting dataset is compatible with downstream modeling and feature extraction pipelines.

- After filtering, source-specific labels are mapped onto the pivot table using a standardized encoding scheme (positive, negative, unlabeled, unknown). Label consistency across sources is then evaluated by counting per-sequence label occurrences, computing positive and negative vote percentages, and deriving high-level classification flags (exclusive positive, exclusive negative, unlabeled-only, or ambiguous). Sequences with conflicting evidence are explicitly identified and stratified based on the proportion of positive annotations.

- As output, the notebook produces multiple non-overlapping sequence subsets, including only-positive, only-negative, and ambiguous cytolysis annotations, along with a comprehensive metadata file summarizing filtering statistics, label agreement, and dataset composition. All outputs are exported in a reproducible format suitable for downstream machine learning, benchmarking, and comparative analysis

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/cytolysis"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_AMPDB_cytolysis = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDB v1/processed_cytolysis_dataset.csv")
df_AMPDB_cytolysis = df_AMPDB_cytolysis.rename(columns={"label": "cytolysis"})

- Collecting all sequences for activity

In [4]:
unique_sequence = count_unique_sequence([df_AMPDB_cytolysis])

1651


- Create pivote dataset

In [5]:
df_pivote = create_pivote(unique_sequence)

- Removing sequences with non canonical residues 

In [6]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [7]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     1645
False       6
Name: count, dtype: int64


- Filter sequences by length

In [8]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    1645.000000
mean      150.957447
std        87.783704
min        11.000000
25%       135.000000
50%       154.000000
75%       167.000000
max       763.000000
Name: length, dtype: float64

In [9]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [10]:
df_pivote["filter_length"].value_counts()

filter_length
False    1347
True      298
Name: count, dtype: int64

In [11]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [12]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(298, 4)

In [13]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [14]:
df_list = [("AMPDB", df_AMPDB_cytolysis)]

In [15]:
for source, dataset in df_list:
    dataset = dataset[["sequence", "cytolysis"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["cytolysis"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [16]:
df_pivote.head(5)

,sequence,AMPDB
0,GIFGKILGAGKKVLCGLSGLC,1
4,KDGYLVGNDGCKYSCFTRPGTYCANECSRVKGKDGYCYAWMACYCY...,1
14,FLGSLFSIGSKLLPGVIKLFQRKKQ,1
16,MKTQFAVLIISMILMQMLVQTEAGFWGKLWEGVKSAIGKRSLRNQD...,1
17,SIGSAFKKALPVAKKIGKAALPIAKAALP,1


- Working with pivote for detecting ambiguous sequences 

In [17]:
df_pivote = process_count_labels(df_pivote) # Verify the consistency of the labels by source

In [18]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
False    298
Name: count, dtype: int64

In [19]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
False    298
Name: count, dtype: int64

In [20]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
True    298
Name: count, dtype: int64

In [21]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
True    298
Name: count, dtype: int64

In [22]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    298
Name: count, dtype: int64

In [23]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,AMPDB,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
0,GIFGKILGAGKKVLCGLSGLC,1,1,0,0,0,True,False,True,False,False,0.0,100.0
4,KDGYLVGNDGCKYSCFTRPGTYCANECSRVKGKDGYCYAWMACYCY...,1,1,0,0,0,True,False,True,False,False,0.0,100.0
14,FLGSLFSIGSKLLPGVIKLFQRKKQ,1,1,0,0,0,True,False,True,False,False,0.0,100.0
16,MKTQFAVLIISMILMQMLVQTEAGFWGKLWEGVKSAIGKRSLRNQD...,1,1,0,0,0,True,False,True,False,False,0.0,100.0
17,SIGSAFKKALPVAKKIGKAALPIAKAALP,1,1,0,0,0,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622,LFGFLIPLLPHLIGAIPQVIGAIR,1,1,0,0,0,True,False,True,False,False,0.0,100.0
1627,GIFSLVKGAAKLAGKGLAKEGGKFGLELIACKIAKQC,1,1,0,0,0,True,False,True,False,False,0.0,100.0
1633,IVPFLLGMVPKLVCLITKKC,1,1,0,0,0,True,False,True,False,False,0.0,100.0
1637,FLPIPRPILLGLL,1,1,0,0,0,True,False,True,False,False,0.0,100.0


- Splitting data into only negative, only positive, and with amiguous data

In [24]:
negative = df_pivote[df_pivote["negative"]]

In [25]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [26]:
positive = df_pivote[df_pivote["positive"]]

In [27]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [28]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [29]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [30]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [31]:
df_ambiguous["Category_pbb"].value_counts()

Series([], Name: count, dtype: int64)

- Working with metada

In [32]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="cytolysis",
    source_list=df_list,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)

metadata

{'task': 'cytolysis',
 'generated_at': '2026-09-04T20:36:00.112225',
 'sources': {'n_unique_sequences': {'AMPDB': 1651}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 1651, 'after': 1645},
  'length_filter': {'before': 1645, 'after': 298},
  'length_distribution': {'min': 11,
   'max': 70,
   'mean': 39.35,
   'median': 30.0}},
 'statistics': {'total_sequences_final': 298,
  'positive': {'positive_and_unlabel': 298, 'only_positive': 298},
  'negative': {'negative_and_unlabel': 0, 'only_negative': 0},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 0}}}

- Exporting data

In [33]:
os.makedirs(output_folder, exist_ok=True)

In [34]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [35]:
negative.shape

(0, 13)

In [36]:
only_negative.shape

(0, 13)

In [37]:
positive.shape

(298, 13)

In [38]:
only_positive.shape

(298, 13)

In [39]:
only_unlabel.shape

(0, 13)

In [40]:
df_ambiguous.shape

(0, 14)

In [41]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)